In [ ]:
"""
exp01 Step 01: Model Dataset Generation

Objective:

    - Join, for every decision bar, the feature rows (pipeline TSIND step02, TSSEG step03, TSCTX step04) with the barrier
      target row (pipeline TSBAR step05). Pairing: feature row at bar t <-> target row with decision_ts = t (entry at bar t+1).
    - Keep EVERY eligible decision row (no sampling here). Sampling for training is applied when the dataset is read
      (model_dataset.read_model_dataset_pdf), so it can change in the experiment config without regenerating this step.
    - The raw TSBAR cells are kept; labels and costs are parsed when the dataset is read.

No trial is logged here (no result is computed). The preview cells only show the exploration period (2005-2014).
Prerequisites: pipeline steps 01 to 05. Outputs: store04_experiments/exp01_minute_entry/step01_model_dataset_data/
(one file per session, model_dataset_YYYYMMDD.csv; about 5,400 files on a first run).
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_date_range_file_path_list, generate_func_data_date_list, get_missing_date_list
# IMPORT BARRIER LABEL FUNCTIONS
from so.features.barrier_labels import get_TSBAR_incomplete_date_list
# IMPORT MODEL DATASET FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import get_date_model_dataset_pdf, read_model_dataset_pdf, get_model_feature_col_str_list
# IMPORT TRIAL LOG FUNCTIONS (CONFIGURATION HASH ONLY: NOTHING IS LOGGED IN THIS STEP)
from so.core.trial_log import get_config_hash_str

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data")
print(f"Output folder:\t{output_path_str}")

In [ ]:
# COLLECT THE TSBAR DATE RANGE (THE DATASET CANNOT EXTEND BEYOND THE TARGETS)
TSBAR_file_path_list = get_date_range_file_path_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR)
start_date_str = pd.to_datetime(TSBAR_file_path_list[0].split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d")
end_date_str = pd.to_datetime(TSBAR_file_path_list[-1].split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d")
# DISPLAY INFORMATION
print(f"TSBAR Date Range:\t{start_date_str} -> {end_date_str}\t({len(TSBAR_file_path_list):,} files)")

In [ ]:
# DEFINE A SAMPLE DATE IN THE EXPLORATION PERIOD (THE LAST TSBAR DATE OF 2014)
sample_date_str = max(pd.to_datetime(f.split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d") for f in TSBAR_file_path_list
                      if pd.to_datetime(f.split("_")[-1].replace(".csv", "")) <= pd.Timestamp(config.EXPLORATION_END_DATE_STR))
# CALL FUNCTION TO GENERATE THE SAMPLE DATE MODEL DATASET
sample_date_dataset_pdf = get_date_model_dataset_pdf(sample_date_str)
# DISPLAY THE FEATURE NULL COUNTS (EARLY-SESSION INDICATORS SHOULD BE AVAILABLE BY 09:59)
feature_col_str_list = [col for col in config.FEATURE_REGISTRY_DICT if col in sample_date_dataset_pdf.columns]
print(f"Sample date:\t{sample_date_str}")
display(sample_date_dataset_pdf[feature_col_str_list].isna().sum().loc[lambda s: s > 0].reset_index().rename(columns={"index": "feature", 0: "null_count"}))
# PREVIEW DATAFRAME
sample_date_dataset_pdf

In [ ]:
# COLLECT THE DATES WHOSE TARGETS ARE STILL INCOMPLETE (THEY CHANGE WHEN NEW DATA ARRIVES)
incomplete_date_list = get_TSBAR_incomplete_date_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, recent_file_count_in=30)
# DISPLAY INFORMATION
print(f"Incomplete TSBAR Dates:\t{len(incomplete_date_list)}")
# CALL FUNCTION TO REGENERATE THE 30 MOST RECENT DATES (OVERWRITE MODE; COVERS DATES WHOSE TARGETS WERE JUST REGENERATED IN STEP 05)
recent_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in TSBAR_file_path_list[-30:]]
generate_func_data_date_list(get_date_model_dataset_pdf, recent_date_list, output_path_str, "model_dataset", "W")

In [ ]:
# GET MISSING DATE LIST (EVERY DATE ON A FIRST RUN)
missing_date_list = get_missing_date_list(output_path_str, start_date_str, end_date_str)
# DISPLAY INFORMATION
print(f"Missing dates:\t{len(missing_date_list):,}")
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST (IGNORE MODE: EXISTING FILES ARE KEPT)
generate_func_data_date_list(get_date_model_dataset_pdf, missing_date_list, output_path_str, "model_dataset", "I")

In [ ]:
# CALL FUNCTION TO READ THE LAST 3 MONTHS OF THE EXPLORATION PERIOD WITH THE TRAINING SAMPLING AND THE MODEL DELTA LABELS
preview_end_str = config.EXPLORATION_END_DATE_STR
preview_dataset_pdf = read_model_dataset_pdf(str((pd.to_datetime(preview_end_str) - pd.DateOffset(months=3)).date()), preview_end_str,
                                             exp_config.MODEL_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# DISPLAY INFORMATION
print(f"Sampled Rows:\t{len(preview_dataset_pdf):,}\t(mode: {exp_config.SAMPLING_MODE}, every {exp_config.SAMPLE_EVERY_N_MINUTES} minutes)")
print(f"Model Features ({len(get_model_feature_col_str_list(preview_dataset_pdf))}):\t{get_model_feature_col_str_list(preview_dataset_pdf)}")
# PREVIEW DATAFRAME
preview_dataset_pdf